# 02 — Silver Transform
**Pipeline:** Bronze Delta → clean + enrich → Silver Delta  
**Layer:** Silver (typed, enriched, feature-engineered records)  
**Runs on:** Databricks cluster (Unity Catalog workspace)  
**Called by:** `flows/trade_pipeline_flow.py` → `transform_silver` task

## Cell 1 — Install dependencies & set up repo path

In [ ]:
%pip install delta-spark --quiet

import sys
import os

REPO_ROOT = "/Workspace/Users/pravash.panigrahi07@gmail.com/trade-analytics-lakehouse"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Repo root: {REPO_ROOT}")
print(f"Python: {sys.version}")

## Cell 2 — Verify config imports and resolved paths

In [ ]:
from src.trade_analytics.config.enums import SPARK_ENV, StorageLayer
from src.trade_analytics.config.settings import (
    BRONZE_PATH,
    SILVER_PATH,
    LARGE_TRADE_THRESHOLD_EUR,
    VELOCITY_MAX_TRADES,
    FX_RATES_TO_EUR,
)
from src.trade_analytics.config.utils import configure_adls_auth

print(f"SPARK_ENV                 : {SPARK_ENV}")
print(f"BRONZE_PATH               : {BRONZE_PATH}")
print(f"SILVER_PATH               : {SILVER_PATH}")
print(f"LARGE_TRADE_THRESHOLD_EUR : {LARGE_TRADE_THRESHOLD_EUR:,}")
print(f"VELOCITY_MAX_TRADES       : {VELOCITY_MAX_TRADES}")
print(f"FX_RATES_TO_EUR           : {FX_RATES_TO_EUR}")

configure_adls_auth(spark)

## Cell 3 — Verify Bronze table exists and is readable

In [ ]:
df_bronze = spark.read.format("delta").load(BRONZE_PATH)

print(f"[Bronze] Row count : {df_bronze.count():,}")
print(f"[Bronze] Schema    :")
df_bronze.printSchema()

print("\n[Bronze] Sample rows:")
display(df_bronze.limit(3))

## Cell 4 — Import and run the full Silver transform

In [ ]:
# Import the transform function from the Python module
# This keeps notebook and script logic in sync — one source of truth
from local_data_transformations.silver_transform import (
    _cast_and_clean,
    _apply_business_filters,
    _engineer_features,
    _select_silver_schema,
)

print("Silver transform functions imported ✓")

## Cell 5 — Step 1: Cast and clean

In [ ]:
df = _cast_and_clean(df_bronze)

print("\nSchema after cast & clean:")
df.printSchema()

print("\nSample — timestamp and derived date/hour columns:")
display(df.select(
    "trade_id", "trade_timestamp", "trade_date", "trade_hour", "trade_minute"
).limit(5))

## Cell 6 — Step 2: Apply business filters

In [ ]:
df = _apply_business_filters(df)

print("\nStatus breakdown after filter (CANCELLED removed):")
display(df.groupBy("status").count().orderBy("status"))

## Cell 7 — Step 3: Engineer features

In [ ]:
df = _engineer_features(df, spark)

print("Schema after feature engineering:")
df.printSchema()

print("\nFX conversion spot check (notional vs notional_eur):")
display(df.select(
    "trade_id", "instrument", "direction",
    "notional", "notional_eur", "signed_notional_eur"
).limit(5))

print("\nRisk tier distribution:")
display(df.groupBy("risk_tier").count().orderBy("risk_tier"))

print("\nAlert level distribution:")
display(df.groupBy("alert_level").count().orderBy("alert_level"))

print("\nVelocity-flagged traders (top 10):")
from pyspark.sql import functions as F
display(
    df.filter(F.col("velocity_flag") == True)
      .groupBy("trader_id")
      .agg(F.count("trade_id").alias("flagged_count"))
      .orderBy(F.col("flagged_count").desc())
      .limit(10)
)

## Cell 8 — Step 4: Select final Silver schema

In [ ]:
df_silver = _select_silver_schema(df)

print(f"Final Silver column count : {len(df_silver.columns)}")
print(f"Final Silver row count    : {df_silver.count():,}")
print("\nFinal Silver schema:")
df_silver.printSchema()

## Cell 9 — Write Silver Delta (partitioned + ZORDER)

In [ ]:
print(f"Writing Silver Delta to: {SILVER_PATH}")

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("trade_date")
    .save(SILVER_PATH)
)

row_count = spark.read.format("delta").load(SILVER_PATH).count()
print(f"[Silver] Write complete. Row count: {row_count:,} ✓")

# OPTIMIZE + ZORDER for downstream query performance
try:
    spark.sql(f"""
        OPTIMIZE delta.`{SILVER_PATH}`
        ZORDER BY (trader_id, instrument)
    """)
    print("[Silver] OPTIMIZE + ZORDER complete ✓")
except Exception as e:
    print(f"[Silver] OPTIMIZE skipped: {e}")

## Cell 10 — Register Silver as Unity Catalog table

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS trade_analytics")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS trade_analytics.silver_trades_enriched
    USING DELTA
    LOCATION '{SILVER_PATH}'
""")

print("Silver table registered in Unity Catalog ✓")
display(spark.sql("SHOW TABLES IN trade_analytics"))

## Cell 11 — Final Silver summary

In [ ]:
df_check = spark.read.format("delta").load(SILVER_PATH)

print("\n── Silver Summary ────────────────────────────────────")
print(f"  Total rows          : {df_check.count():,}")
print(f"  Date range          : "
      f"{df_check.selectExpr('min(trade_date)').collect()[0][0]}"
      f" → {df_check.selectExpr('max(trade_date)').collect()[0][0]}")
print(f"  Distinct traders    : {df_check.select('trader_id').distinct().count():,}")
print(f"  Distinct instruments: {df_check.select('instrument').distinct().count():,}")

print("\n  Partition sizes (rows per trade_date):")
display(
    df_check.groupBy("trade_date")
            .count()
            .orderBy("trade_date")
)

print("\n  Delta table history:")
display(spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_PATH}`"))

print("\nSilver transform complete ✓")